# OGBN-Arxiv Data Mining Notebook (Kaggle Ready)

Notebook ini fokus pada **data mining dan preprocessing** dataset **`ogbn-arxiv`** sampai **siap dipakai untuk training**, **tanpa** membangun atau melatih model.

Cakupan notebook:
1. Mengunduh dataset `ogbn-arxiv`
2. Memahami struktur graph dan data mentah
3. Melakukan EDA pada node, edge, tahun, label, dan fitur
4. Menjelaskan hubungan antar-node
5. Menjelaskan arti fitur node
6. Menangani data hingga siap training secara benar (tanpa leakage)
7. Menyimpan artefak preprocessing untuk tahap modeling berikutnya

> **Catatan penting tentang fitur node:**  
> Pada `ogbn-arxiv`, setiap node adalah **paper arXiv bidang Computer Science**, dan setiap edge terarah menunjukkan bahwa **satu paper mengutip paper lain**.  
> Fitur node **bukan** kolom yang bisa dinamai satu per satu seperti `author_count`, `page_count`, dan seterusnya.  
> Setiap node punya **vektor 128 dimensi** yang dibentuk dari **rata-rata embedding kata** pada **judul dan abstrak** paper. Jadi, tiap dimensi adalah komponen embedding laten, **bukan fitur semantik yang punya nama manusiawi**.

## Referensi Konsep Dataset

Secara resmi, OGB mendeskripsikan `ogbn-arxiv` sebagai:
- graph **terarah**
- **169,343 node**
- **1,166,243 edge**
- tugas **40-class node classification**
- split data berbasis waktu:
  - train: paper hingga **2017**
  - valid: paper **2018**
  - test: paper sejak **2019**

Implikasi data mining:
- relasi sitasi sangat penting karena node tidak berdiri sendiri
- struktur graph + fitur konten saling melengkapi
- preprocessing **harus menjaga split waktu** agar tidak leakage

In [ ]:
# Jika menjalankan di Kaggle:
# - Pastikan Internet ON jika ingin dataset diunduh otomatis dari OGB
# - Jika Internet OFF, Anda bisa attach dataset yang sudah pernah diunduh sebelumnya

!pip -q install ogb networkx plotly

In [ ]:
import os
import gc
import gzip
import json
import math
import shutil
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

from ogb.nodeproppred import NodePropPredDataset

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
np.set_printoptions(suppress=True)

## 1) Download dan Load Dataset OGB

Di sini kita memakai **library-agnostic loader** agar preprocessing mudah dipahami.
Output utama:
- `graph['edge_index']`: pasangan source-target untuk edge
- `graph['node_feat']`: matriks fitur node
- `graph['node_year']`: metadata tahun publikasi node
- `labels`: label kelas tiap node

In [ ]:
DATA_ROOT = Path("/kaggle/working/ogb_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

dataset = NodePropPredDataset(name="ogbn-arxiv", root=str(DATA_ROOT))
split_idx = dataset.get_idx_split()
graph, labels = dataset[0]

train_idx = split_idx["train"]
valid_idx = split_idx["valid"]
test_idx  = split_idx["test"]

edge_index = graph["edge_index"]       # shape: (2, num_edges)
node_feat = graph["node_feat"]         # shape: (num_nodes, 128)
node_year = graph["node_year"]         # shape: (num_nodes, 1)
num_nodes = graph["num_nodes"]
num_edges = edge_index.shape[1]
y = labels.reshape(-1)

print("num_nodes :", num_nodes)
print("num_edges :", num_edges)
print("node_feat :", node_feat.shape)
print("node_year :", node_year.shape)
print("labels    :", labels.shape)
print("train/valid/test :", len(train_idx), len(valid_idx), len(test_idx))

## 2) Memahami Folder Data Mentah yang Diunduh OGB

OGB biasanya menyimpan file hasil unduhan dan pemrosesan awal di bawah folder root.
Cell berikut membantu melihat struktur file agar kita paham apa yang tersedia.

In [ ]:
for path in sorted(DATA_ROOT.rglob("*")):
    depth = len(path.relative_to(DATA_ROOT).parts)
    indent = "  " * depth
    print(f"{indent}{path.name}")

### Apa arti struktur data ini?

Umumnya Anda akan menemukan hal seperti:
- folder `raw/` berisi file graph mentah
- folder `mapping/` berisi pemetaan indeks node ke paper ID asli
- file split atau processed data dari OGB

Untuk `ogbn-arxiv`, file penting secara konsep adalah:
- edge list sitasi
- fitur node 128 dimensi
- tahun publikasi node
- label kelas
- mapping `nodeidx -> paperid`

## 3) Bangun Tabel Node-Level untuk EDA

Kita ubah graph menjadi tabel node supaya analisis data lebih mudah.

In [ ]:
node_df = pd.DataFrame({
    "node_id": np.arange(num_nodes, dtype=np.int64),
    "year": node_year.reshape(-1).astype(np.int64),
    "label": y.astype(np.int64),
})

node_df["split"] = "unassigned"
node_df.loc[train_idx, "split"] = "train"
node_df.loc[valid_idx, "split"] = "valid"
node_df.loc[test_idx,  "split"] = "test"

node_df.head()

## 4) Bangun Tabel Edge-Level dan Jelaskan Hubungan Antar Node

`edge_index` berbentuk `(2, num_edges)`:
- baris pertama = **source**
- baris kedua = **target**

Untuk `ogbn-arxiv`, edge terarah berarti:

**source paper -> target paper**  
artinya **paper sumber mengutip paper target**.

Jadi relasi antar node adalah **relasi sitasi ilmiah**, bukan relasi sosial, jarak, transaksi, atau kemiripan.

In [ ]:
edge_df = pd.DataFrame({
    "src": edge_index[0].astype(np.int64),
    "dst": edge_index[1].astype(np.int64),
})

edge_df.head()

In [ ]:
# Tambahkan informasi tahun source dan target untuk analisis
edge_df["src_year"] = node_df.loc[edge_df["src"], "year"].to_numpy()
edge_df["dst_year"] = node_df.loc[edge_df["dst"], "year"].to_numpy()

edge_df["year_diff_src_minus_dst"] = edge_df["src_year"] - edge_df["dst_year"]
edge_df.head()

### Interpretasi hubungan node

Beberapa analisis relasi yang berguna:
- **in-degree** tinggi: paper sering dikutip paper lain
- **out-degree** tinggi: paper mengutip banyak referensi
- **year difference**:
  - biasanya paper baru mengutip paper lama
  - jika source lebih tua dari target, perlu dicek karena bisa berasal dari metadata/konstruksi graph tertentu

In [ ]:
in_degree = np.bincount(edge_df["dst"], minlength=num_nodes)
out_degree = np.bincount(edge_df["src"], minlength=num_nodes)

node_df["in_degree"] = in_degree
node_df["out_degree"] = out_degree
node_df["total_degree"] = node_df["in_degree"] + node_df["out_degree"]

node_df[["in_degree", "out_degree", "total_degree"]].describe().T

## 5) EDA Dasar: Missing Value, Duplikasi, Self-loop

Langkah awal data mining adalah memastikan kualitas data graph.

In [ ]:
print("Missing feature values :", np.isnan(node_feat).sum())
print("Missing labels         :", pd.isna(node_df["label"]).sum())
print("Missing years          :", pd.isna(node_df["year"]).sum())

print("Duplicate edges        :", edge_df.duplicated().sum())
print("Self loops             :", (edge_df["src"] == edge_df["dst"]).sum())

## 6) Distribusi Split dan Label

Karena dataset memakai split berbasis waktu, kita perlu melihat distribusi jumlah node per split dan kelas.

In [ ]:
split_counts = node_df["split"].value_counts().rename_axis("split").reset_index(name="count")
split_counts

In [ ]:
fig = px.bar(split_counts, x="split", y="count", title="Jumlah Node per Split")
fig.show()

In [ ]:
label_counts = node_df["label"].value_counts().sort_index().rename_axis("label").reset_index(name="count")
label_counts.head()

In [ ]:
fig = px.bar(label_counts, x="label", y="count", title="Distribusi Label (40 kelas)")
fig.show()

### Insight yang dicari dari distribusi label
- apakah ada **class imbalance**
- apakah ada kelas yang sangat jarang
- apakah distribusi kelas berubah antar split waktu

Ini penting karena pada tahap modeling nanti bisa memengaruhi strategi evaluasi, weighting, atau sampling.

In [ ]:
label_by_split = (
    node_df.groupby(["split", "label"])
    .size()
    .reset_index(name="count")
)

fig = px.histogram(
    label_by_split,
    x="label",
    y="count",
    color="split",
    barmode="group",
    title="Distribusi Label per Split"
)
fig.show()

## 7) EDA Waktu Publikasi

Split resmi OGB berbasis waktu. Karena itu, tahun publikasi adalah variabel penting untuk:
- memahami drift data
- memvalidasi split
- mencegah leakage

In [ ]:
year_counts = node_df["year"].value_counts().sort_index().rename_axis("year").reset_index(name="count")
year_counts.head()

In [ ]:
fig = px.line(year_counts, x="year", y="count", markers=True, title="Jumlah Paper per Tahun")
fig.show()

In [ ]:
split_year_summary = (
    node_df.groupby("split")["year"]
    .agg(["min", "max", "median", "nunique", "count"])
    .reset_index()
)
split_year_summary

### Validasi split waktu
Secara konsep OGB menyarankan:
- train hingga 2017
- valid pada 2018
- test sejak 2019

Mari cek apakah split node memang konsisten terhadap tahun.

In [ ]:
pd.crosstab(node_df["year"], node_df["split"])

## 8) EDA Struktur Graph: Degree Distribution

Citation network biasanya sangat skewed:
- sebagian kecil paper sangat populer dan banyak dikutip
- sebagian besar paper hanya punya sedikit sitasi

In [ ]:
degree_summary = node_df[["in_degree", "out_degree", "total_degree"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
degree_summary

In [ ]:
fig = px.histogram(node_df, x="in_degree", nbins=100, title="Distribusi In-Degree")
fig.show()

fig = px.histogram(node_df, x="out_degree", nbins=100, title="Distribusi Out-Degree")
fig.show()

In [ ]:
# Visualisasi log1p agar tail lebih terlihat
node_df["log1p_in_degree"] = np.log1p(node_df["in_degree"])
node_df["log1p_out_degree"] = np.log1p(node_df["out_degree"])

fig = px.histogram(node_df, x="log1p_in_degree", nbins=100, title="Distribusi log1p(In-Degree)")
fig.show()

fig = px.histogram(node_df, x="log1p_out_degree", nbins=100, title="Distribusi log1p(Out-Degree)")
fig.show()

## 9) EDA Relasi Tahun Source-Target

Dalam graph sitasi, lazimnya paper yang lebih baru mengutip paper yang lebih lama.

In [ ]:
edge_df["relation_type_by_year"] = np.select(
    [
        edge_df["src_year"] > edge_df["dst_year"],
        edge_df["src_year"] == edge_df["dst_year"],
        edge_df["src_year"] < edge_df["dst_year"],
    ],
    [
        "newer_src_cites_older_dst",
        "same_year_citation",
        "older_src_cites_newer_dst",
    ],
    default="unknown"
)

edge_df["relation_type_by_year"].value_counts()

In [ ]:
fig = px.histogram(
    edge_df.sample(min(len(edge_df), 200000), random_state=42),
    x="year_diff_src_minus_dst",
    nbins=100,
    title="Distribusi (src_year - dst_year) pada sampel edge"
)
fig.show()

### Makna analitis
- nilai **positif**: source lebih baru dari target, sesuai pola sitasi umum
- nilai **nol**: sitasi ke paper tahun yang sama
- nilai **negatif**: source lebih tua dari target, menarik untuk dicek lebih lanjut

Perlu diingat: pada dataset benchmark, orientasi edge mengikuti definisi graph resminya. Untuk modeling tertentu, sebagian peneliti menambahkan reverse edge atau menyimetriskan graph.

## 10) EDA Fitur Node

### Apa arti tiap fitur node?

Setiap node punya **128 fitur laten** hasil rata-rata embedding kata dari judul dan abstrak paper.

Jadi:
- `feat_0`, `feat_1`, ..., `feat_127` **tidak punya nama fitur semantik**
- tiap dimensi adalah koordinat dalam ruang embedding
- interpretasinya lebih cocok secara statistik, bukan secara manual per fitur

Pendekatan yang masuk akal:
1. lihat shape dan tipe data
2. cek missing value / constant feature
3. lihat statistik tiap dimensi
4. lihat reduksi dimensi (PCA) untuk pola global

In [ ]:
feature_df = pd.DataFrame(node_feat, columns=[f"feat_{i}" for i in range(node_feat.shape[1])])
feature_df.head()

In [ ]:
feature_stats = feature_df.describe().T
feature_stats.head(10)

In [ ]:
feature_quality = pd.DataFrame({
    "feature": feature_df.columns,
    "missing_count": feature_df.isna().sum().values,
    "n_unique": feature_df.nunique(dropna=False).values,
    "std": feature_df.std().values,
    "mean": feature_df.mean().values,
})
feature_quality.head(10)

In [ ]:
constant_features = feature_quality.query("std == 0")
constant_features.head()

### Penjelasan praktis tentang fitur node
Karena fitur adalah embedding:
- dimensi individual **tidak bisa dijelaskan** sebagai “jumlah kata”, “jumlah sitasi”, atau “topik tertentu”
- interpretasi lebih kuat bila melihat:
  - **jarak antar-node** di ruang fitur
  - **cluster**
  - **arah komponen utama** hasil PCA
  - korelasi fitur dengan label / tahun / degree

In [ ]:
# PCA hanya untuk EDA dan visualisasi, bukan mengganti fitur asli secara default
sample_size = min(50000, num_nodes)
sample_idx = np.random.default_rng(42).choice(num_nodes, size=sample_size, replace=False)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(node_feat[sample_idx])

pca_df = pd.DataFrame({
    "pc1": X_pca[:, 0],
    "pc2": X_pca[:, 1],
    "label": y[sample_idx],
    "year": node_year[sample_idx].reshape(-1),
    "split": node_df.loc[sample_idx, "split"].to_numpy(),
})

print("Explained variance ratio:", pca.explained_variance_ratio_)
pca_df.head()

In [ ]:
pca_plot_df = pca_df.sample(min(len(pca_df), 15000), random_state=42)

fig = px.scatter(
    pca_plot_df,
    x="pc1",
    y="pc2",
    color="split",
    title="Proyeksi PCA 2D dari fitur node (sample)"
)
fig.show()

In [ ]:
pca_label_plot_df = pca_df.sample(min(len(pca_df), 15000), random_state=42).copy()
pca_label_plot_df["label_str"] = pca_label_plot_df["label"].astype(str)

fig = px.scatter(
    pca_label_plot_df,
    x="pc1",
    y="pc2",
    color="label_str",
    title="PCA 2D diwarnai label (sample)"
)
fig.show()

> **Catatan:** PCA hanya membantu melihat pola global.  
> Untuk preprocessing siap training, biasanya fitur asli 128 dimensi tetap dipakai, lalu distandardisasi dengan benar.

## 11) Opsional: Menghubungkan Node dengan Judul dan Abstrak Asli

OGB menyediakan:
- file mapping `nodeidx2paperid.csv.gz` di folder dataset
- file `titleabs.tsv.gz` yang memetakan `paperid -> title, abstract`

Langkah ini **opsional**, tetapi sangat membantu bila Anda ingin:
- menjelaskan contoh node nyata
- memeriksa paper dengan degree tinggi
- melakukan analisis text mining lanjutan

In [ ]:
# Cari file mapping nodeidx -> paperid yang diunduh bersama dataset
mapping_candidates = list(DATA_ROOT.rglob("nodeidx2paperid.csv.gz"))
mapping_candidates[:5]

In [ ]:
if mapping_candidates:
    mapping_path = mapping_candidates[0]
    node_map_df = pd.read_csv(mapping_path, compression="gzip")
    display(node_map_df.head())
    print("Mapping shape:", node_map_df.shape)
else:
    node_map_df = None
    print("File mapping tidak ditemukan.")

In [ ]:
# Opsional: download raw titles & abstracts dari OGB/SNAP
TITLEABS_URL = "https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz"
TITLEABS_PATH = DATA_ROOT / "titleabs.tsv.gz"

if not TITLEABS_PATH.exists():
    try:
        urllib.request.urlretrieve(TITLEABS_URL, TITLEABS_PATH)
        print("Downloaded:", TITLEABS_PATH)
    except Exception as e:
        print("Gagal download titleabs.tsv.gz:", e)
        print("Jika di Kaggle Internet OFF, unggah file ini manual atau skip bagian ini.")
else:
    print("Sudah ada:", TITLEABS_PATH)

In [ ]:
if TITLEABS_PATH.exists():
    titleabs_df = pd.read_csv(
        TITLEABS_PATH,
        sep="\t",
        header=None,
        names=["paper_id", "title", "abstract"],
        compression="gzip"
    )
    display(titleabs_df.head())
    print("titleabs shape:", titleabs_df.shape)
else:
    titleabs_df = None
    print("titleabs.tsv.gz belum tersedia.")

In [ ]:
# Merge node -> paper_id -> title/abstract bila file tersedia
if node_map_df is not None:
    # Menangani variasi nama kolom
    cols_lower = {c.lower(): c for c in node_map_df.columns}
    node_col = cols_lower.get("node idx", None) or cols_lower.get("node_idx", None) or cols_lower.get("nodeid", None)
    paper_col = cols_lower.get("paper id", None) or cols_lower.get("paper_id", None) or cols_lower.get("paperid", None)

    if node_col is None:
        # fallback: biasanya kolom pertama adalah node index
        node_col = node_map_df.columns[0]
    if paper_col is None:
        # fallback: biasanya kolom kedua adalah paper id
        paper_col = node_map_df.columns[1]

    node_map_df = node_map_df.rename(columns={node_col: "node_id", paper_col: "paper_id"})
    node_map_df["node_id"] = node_map_df["node_id"].astype(int)

    node_text_df = node_df.merge(node_map_df[["node_id", "paper_id"]], on="node_id", how="left")

    if titleabs_df is not None:
        node_text_df = node_text_df.merge(titleabs_df, on="paper_id", how="left")

    display(node_text_df.head())
else:
    node_text_df = node_df.copy()
    print("Merge text dilewati karena mapping file tidak tersedia.")

### Mengapa bagian ini penting?
Dengan text mentah:
- Anda bisa menjelaskan node secara lebih nyata
- bisa memeriksa paper yang paling sering dikutip
- bisa menyiapkan pipeline text mining terpisah di masa depan

Namun untuk benchmark OGB standar, fitur yang dipakai tetap **embedding 128 dimensi** yang sudah disediakan.

In [ ]:
if "title" in node_text_df.columns:
    rich_example = (
        node_text_df.sort_values(["in_degree", "out_degree"], ascending=False)
        .head(10)[["node_id", "paper_id", "year", "label", "in_degree", "out_degree", "title"]]
    )
    rich_example
else:
    node_df.sort_values(["in_degree", "out_degree"], ascending=False).head(10)

## 12) Cek Kualitas Split dan Potensi Leakage

Dalam data mining yang benar, transformasi statistik seperti:
- imputasi
- scaling
- PCA (jika dipakai untuk modeling)

harus di-**fit hanya pada training set**, lalu di-apply ke valid/test.

In [ ]:
train_mask = np.zeros(num_nodes, dtype=bool)
valid_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)

train_mask[train_idx] = True
valid_mask[valid_idx] = True
test_mask[test_idx] = True

assert train_mask.sum() == len(train_idx)
assert valid_mask.sum() == len(valid_idx)
assert test_mask.sum() == len(test_idx)
assert (train_mask & valid_mask).sum() == 0
assert (train_mask & test_mask).sum() == 0
assert (valid_mask & test_mask).sum() == 0

print("Mask OK")

## 13) Preprocessing Fitur Node Sampai Siap Training

Strategi preprocessing yang aman:
1. ubah tipe data menjadi konsisten
2. imputasi missing value bila perlu
3. scaling **fit hanya di train**
4. simpan hasil preprocessing

In [ ]:
X_raw = node_feat.astype(np.float32)
y_raw = y.astype(np.int64)
years = node_year.reshape(-1).astype(np.int64)

print(X_raw.dtype, y_raw.dtype, years.dtype)

In [ ]:
# Imputasi: defensif. Pada ogbn-arxiv umumnya tidak ada missing feature,
# tetapi tetap disiapkan agar pipeline robust.
imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_raw[train_idx])
X_valid_imputed = imputer.transform(X_raw[valid_idx])
X_test_imputed  = imputer.transform(X_raw[test_idx])

# Satukan kembali sesuai urutan node asli
X_imputed = np.empty_like(X_raw, dtype=np.float32)
X_imputed[train_idx] = X_train_imputed.astype(np.float32)
X_imputed[valid_idx] = X_valid_imputed.astype(np.float32)
X_imputed[test_idx]  = X_test_imputed.astype(np.float32)

print(X_imputed.shape, X_imputed.dtype)

In [ ]:
# Standardisasi hanya berdasarkan training split
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_imputed[train_idx])
X_valid_scaled = scaler.transform(X_imputed[valid_idx])
X_test_scaled  = scaler.transform(X_imputed[test_idx])

X_scaled = np.empty_like(X_imputed, dtype=np.float32)
X_scaled[train_idx] = X_train_scaled.astype(np.float32)
X_scaled[valid_idx] = X_valid_scaled.astype(np.float32)
X_scaled[test_idx]  = X_test_scaled.astype(np.float32)

print("Scaled feature shape:", X_scaled.shape)
print("Train mean (approx 0):", np.round(X_scaled[train_idx].mean(axis=0)[:5], 4))
print("Train std  (approx 1):", np.round(X_scaled[train_idx].std(axis=0)[:5], 4))

### Kenapa scaling perlu?
Walau GNN tertentu kadang bisa bekerja tanpa scaling, dari perspektif data mining dan pipeline yang rapi:
- scaling membuat distribusi fitur lebih stabil
- membantu metode non-graph baseline juga
- mempermudah eksperimen lanjut

Karena split bersifat waktu, scaling **harus** belajar dari train saja.

## 14) Preprocessing Struktur Graph

Untuk modeling graph nanti, biasanya ada beberapa opsi:
1. **Pertahankan graph terarah** seperti data asli
2. **Tambahkan reverse edge** agar informasi mengalir dua arah
3. **Buat graph undirected** (symmetrize)

Di notebook ini kita **tidak melatih model**, tetapi kita siapkan ketiga opsi agar fleksibel untuk tahap berikutnya.

In [ ]:
edge_index_directed = edge_index.astype(np.int64)

# Tambah reverse edge
edge_index_bidirectional = np.concatenate(
    [edge_index_directed, edge_index_directed[::-1]],
    axis=1
)

# Hapus duplikasi pada versi bidirectional
edge_pairs = pd.DataFrame({
    "src": edge_index_bidirectional[0],
    "dst": edge_index_bidirectional[1],
}).drop_duplicates()

edge_index_undirected_unique = edge_pairs[["src", "dst"]].to_numpy().T.astype(np.int64)

print("Directed edges            :", edge_index_directed.shape[1])
print("Bidirectional edges       :", edge_index_bidirectional.shape[1])
print("Undirected unique as bi-dir:", edge_index_undirected_unique.shape[1])

In [ ]:
# Opsi tambahan: membuang self-loop bila nanti diperlukan
def remove_self_loops(edge_idx):
    mask = edge_idx[0] != edge_idx[1]
    return edge_idx[:, mask]

edge_index_directed_noself = remove_self_loops(edge_index_directed)
edge_index_undirected_noself = remove_self_loops(edge_index_undirected_unique)

print("Directed no self-loop   :", edge_index_directed_noself.shape[1])
print("Undirected no self-loop :", edge_index_undirected_noself.shape[1])

### Kapan memakai graph directed vs undirected?
- **Directed**: saat ingin mempertahankan arti sitasi yang asli
- **Undirected / bidirectional**: saat ingin message passing lebih kaya pada model graph tertentu

Untuk **EDA**, directed graph sangat informatif.  
Untuk **preprocessing siap training**, simpan dua versi agar eksperimen modeling lebih fleksibel.

## 15) Feature Engineering Struktural Sederhana (Opsional, Masih Tahap Preprocessing)

Selain fitur teks bawaan, kita dapat menambahkan fitur struktural sederhana:
- in-degree
- out-degree
- total-degree
- log1p degree
- tahun publikasi

Ini masih termasuk preprocessing / feature engineering, belum training.

In [ ]:
struct_feat_df = pd.DataFrame({
    "in_degree": node_df["in_degree"].astype(np.float32),
    "out_degree": node_df["out_degree"].astype(np.float32),
    "total_degree": node_df["total_degree"].astype(np.float32),
    "log1p_in_degree": np.log1p(node_df["in_degree"]).astype(np.float32),
    "log1p_out_degree": np.log1p(node_df["out_degree"]).astype(np.float32),
    "year": node_df["year"].astype(np.float32),
})

struct_feat_df.head()

In [ ]:
# Standardisasi fitur struktural berdasarkan train saja
struct_scaler = StandardScaler()
S_train = struct_scaler.fit_transform(struct_feat_df.iloc[train_idx])
S_valid = struct_scaler.transform(struct_feat_df.iloc[valid_idx])
S_test  = struct_scaler.transform(struct_feat_df.iloc[test_idx])

S_scaled = np.empty((num_nodes, struct_feat_df.shape[1]), dtype=np.float32)
S_scaled[train_idx] = S_train.astype(np.float32)
S_scaled[valid_idx] = S_valid.astype(np.float32)
S_scaled[test_idx]  = S_test.astype(np.float32)

# Gabung dengan embedding asli yang sudah diskalakan
X_ready = np.concatenate([X_scaled, S_scaled], axis=1).astype(np.float32)

print("Embedding only shape     :", X_scaled.shape)
print("Structural feature shape :", S_scaled.shape)
print("Final training-ready X   :", X_ready.shape)

### Catatan feature engineering
`X_ready` berisi:
- 128 fitur embedding teks bawaan OGB
- fitur struktural tambahan hasil data mining

Ini **belum training**, tapi dataset sudah siap dipakai model nantinya.

## 16) Connectivity Analysis (Opsional tapi berguna untuk memahami graph)

Untuk analisis struktur global, kita buat graph `networkx` dari versi undirected-no-self-loop.
Karena graph besar, analisis berat sebaiknya dilakukan seperlunya.

In [ ]:
# Gunakan graph undirected-no-self-loop untuk analisis konektivitas
G = nx.Graph()
G.add_nodes_from(range(num_nodes))
G.add_edges_from(edge_index_undirected_noself.T.tolist())

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Density        :", nx.density(G))

In [ ]:
num_components = nx.number_connected_components(G)
largest_cc_size = len(max(nx.connected_components(G), key=len))

print("Connected components:", num_components)
print("Largest CC size     :", largest_cc_size)

### Interpretasi connectivity
- jika komponen terbesar sangat dominan, berarti sebagian besar paper saling terhubung lewat rantai sitasi
- komponen kecil dapat menunjukkan subbidang yang lebih terisolasi atau paper dengan koneksi sangat minim

## 17) Simpan Artefak Hasil Preprocessing

Tahap ini menghasilkan file siap dipakai untuk modeling berikutnya.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/ogbn_arxiv_preprocessed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / "X_raw.npy", X_raw)
np.save(OUTPUT_DIR / "X_scaled.npy", X_scaled)
np.save(OUTPUT_DIR / "X_ready.npy", X_ready)
np.save(OUTPUT_DIR / "y.npy", y_raw)
np.save(OUTPUT_DIR / "year.npy", years)

np.save(OUTPUT_DIR / "train_idx.npy", train_idx)
np.save(OUTPUT_DIR / "valid_idx.npy", valid_idx)
np.save(OUTPUT_DIR / "test_idx.npy", test_idx)

np.save(OUTPUT_DIR / "edge_index_directed.npy", edge_index_directed)
np.save(OUTPUT_DIR / "edge_index_directed_noself.npy", edge_index_directed_noself)
np.save(OUTPUT_DIR / "edge_index_undirected_noself.npy", edge_index_undirected_noself)

node_df.to_csv(OUTPUT_DIR / "node_table.csv", index=False)
edge_df.head(500000).to_csv(OUTPUT_DIR / "edge_table_sample.csv", index=False)  # simpan sampel agar ukuran wajar

with open(OUTPUT_DIR / "metadata.json", "w") as f:
    json.dump({
        "dataset_name": "ogbn-arxiv",
        "num_nodes": int(num_nodes),
        "num_edges_directed": int(edge_index_directed.shape[1]),
        "node_feature_dim_original": int(X_raw.shape[1]),
        "node_feature_dim_ready": int(X_ready.shape[1]),
        "num_classes": int(len(np.unique(y_raw))),
        "train_size": int(len(train_idx)),
        "valid_size": int(len(valid_idx)),
        "test_size": int(len(test_idx)),
        "notes": {
            "graph_semantics": "src paper cites dst paper",
            "feature_semantics": "128-dimensional averaged word embeddings from title and abstract",
            "preprocessing_rule": "imputer and scaler fitted on train split only",
        }
    }, f, indent=2)

print("Saved files in:", OUTPUT_DIR)
print(sorted([p.name for p in OUTPUT_DIR.iterdir()]))

## 18) Ringkasan Akhir

### Apa itu node?
Node = **paper arXiv CS**

### Apa itu edge?
Edge terarah = **relasi sitasi**
- `src -> dst` berarti paper sumber **mengutip** paper target

### Apa label node?
Label = **kategori utama paper** dari **40 subject area** arXiv CS

### Apa fitur node?
Fitur = **vektor embedding 128 dimensi** dari judul + abstrak  
Artinya:
- tiap dimensi tidak punya nama fitur manusiawi
- interpretasi terbaik dilakukan secara statistik / geometrik

### Apa hasil preprocessing di notebook ini?
Notebook ini menghasilkan:
- tabel node dan edge untuk EDA
- statistik struktur graph
- versi fitur yang sudah diimputasi dan diskalakan
- fitur struktural tambahan
- versi graph directed dan undirected-no-self-loop
- file `.npy`, `.csv`, `.json` yang siap dipakai tahap modeling berikutnya

### Apa yang belum dilakukan?
- belum membangun model
- belum training
- belum evaluasi

Itu memang sengaja, karena notebook ini difokuskan pada **data mining + preprocessing**.

## 19) Checklist Praktis untuk Tahap Modeling Setelah Ini

Setelah notebook ini, Anda bisa lanjut ke notebook kedua untuk:
1. baseline non-graph (mis. Logistic Regression / MLP) dengan `X_scaled` atau `X_ready`
2. model graph (GCN, GraphSAGE, GAT) dengan `edge_index_*`
3. eksperimen perbandingan:
   - directed vs undirected
   - embedding only vs embedding + structural features

Tetapi di notebook ini kita berhenti sampai **data siap training**.